# Обучение двух моделей литотипизации керна (ДС + УФ)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

import torch
import torch.nn as nn
import torch.optim as optim

from src.utils import set_seed
from src.transforms import get_transforms
from src.data import prepare_loaders
from src.models.resnet import create_resnet18
from src.training import train_one_epoch, validate, EarlyStopping, save_history

# ──────────────────────────────────────────────────
# КОНФИГУРАЦИЯ
# ──────────────────────────────────────────────────
DATA_ROOT = Path(r"C:\D\GazProm\nogit\Digital_core_tmv2")
SAVE_DIR  = Path("./models")
SAVE_DIR.mkdir(exist_ok=True)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE   = 7
MIN_DELTA  = 0.005
SEED       = 42

# Лучшие гиперпараметры по итогам экспериментов
CONFIGS = {
    "ДС": dict(lr=5e-5, wd=1e-3, dropout=0.3, freeze="none",    resize="pad",  aug="heavy"),
    "УФ": dict(lr=5e-4, wd=1e-5, dropout=0.3, freeze="partial", resize="crop", aug="std"),
}

print(f"Device : {DEVICE}")
print(f"Epochs : {MAX_EPOCHS}  |  Patience : {PATIENCE}  |  Batch : {BATCH_SIZE}")


## Обучение

In [ ]:
def train_model(modality: str, cfg: dict):
    gen = set_seed(SEED)
    print(f"\n{'='*60}")
    print(f"  Модальность: {modality}")
    print(f"  LR={cfg['lr']:.0e}  WD={cfg['wd']:.0e}  Drop={cfg['dropout']}  Freeze={cfg['freeze']}")
    print(f"  Resize={cfg['resize']}  Aug={cfg['aug']}")
    print(f"{'='*60}")

    train_loader, val_loader, classes = prepare_loaders(
        DATA_ROOT / modality,
        train_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=True),
        val_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=False),
        batch_size=BATCH_SIZE,
        generator=gen,
    )
    print(f"Классов: {len(classes)} | Трейн: {len(train_loader.dataset)} | Вал: {len(val_loader.dataset)}")

    model = create_resnet18(len(classes), freeze_mode=cfg["freeze"], dropout_p=cfg["dropout"]).to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9, weight_decay=cfg["wd"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    es = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, MAX_EPOCHS)
        val_mets   = validate(model, val_loader, criterion, DEVICE, epoch, MAX_EPOCHS)
        scheduler.step()

        improved = es.step(val_mets["f1"], epoch)
        if improved:
            torch.save(model.state_dict(), SAVE_DIR / f"{modality}_best.pth")

        history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_mets.items()}})
        print(f"Ep {epoch:3d} | TrL {train_loss:.4f} | VL {val_mets['loss']:.4f} | "
              f"F1 {val_mets['f1']:.4f} | Acc {val_mets['acc']:.4f} | {es.status}")

        if es.should_stop:
            print(f"\nEarly stop. Лучшая эпоха: {es.best_epoch}  F1={es.best:.4f}")
            break

    save_history(history, SAVE_DIR / f"{modality}_history.json")
    torch.cuda.empty_cache()
    return history, classes


results = {}
for mod, cfg in CONFIGS.items():
    history, classes = train_model(mod, cfg)
    results[mod] = {"history": history, "classes": classes}


## Кривые обучения и метрики

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.transforms import get_transforms
from src.models.resnet import create_resnet18
from src.data import prepare_loaders
from src.utils import set_seed

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for row, (mod, cfg) in enumerate(CONFIGS.items()):
    hist    = results[mod]["history"]
    classes = results[mod]["classes"]
    epochs  = [h["epoch"] for h in hist]

    # — Loss curve —
    ax = fig.add_subplot(gs[row, 0])
    ax.plot(epochs, [h["train_loss"]  for h in hist], label="Train")
    ax.plot(epochs, [h["val_loss"]    for h in hist], label="Val")
    ax.set_title(f"{mod} — Loss"); ax.set_xlabel("Epoch"); ax.legend()

    # — F1 curve —
    ax = fig.add_subplot(gs[row, 1])
    ax.plot(epochs, [h["val_f1"]  for h in hist], color="tab:green")
    ax.set_title(f"{mod} — Val F1"); ax.set_xlabel("Epoch")

    # — Acc / Prec / Rec —
    ax = fig.add_subplot(gs[row, 2])
    ax.plot(epochs, [h["val_acc"]  for h in hist], label="Acc")
    ax.plot(epochs, [h["val_prec"] for h in hist], label="Prec")
    ax.plot(epochs, [h["val_rec"]  for h in hist], label="Rec")
    ax.set_title(f"{mod} — Val metrics"); ax.set_xlabel("Epoch"); ax.legend()

    # — Confusion matrix —
    gen = set_seed(SEED)
    _, val_loader, _ = prepare_loaders(
        DATA_ROOT / mod,
        train_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=True),
        val_transform=get_transforms(cfg["resize"], cfg["aug"], is_train=False),
        batch_size=BATCH_SIZE,
        generator=gen,
    )
    model = create_resnet18(len(classes), freeze_mode=cfg["freeze"], dropout_p=cfg["dropout"]).to(DEVICE)
    model.load_state_dict(torch.load(SAVE_DIR / f"{mod}_best.pth", map_location=DEVICE))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            _, preds = torch.max(model(inputs), 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    ax = fig.add_subplot(gs[row, 3])
    disp = ConfusionMatrixDisplay(cm, display_labels=[c[:6] for c in classes])
    disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(f"{mod} — Confusion matrix")

plt.suptitle("Результаты обучения", fontsize=14, y=1.01)
plt.savefig(SAVE_DIR / "training_results.png", dpi=120, bbox_inches="tight")
plt.show()

# Итоговая таблица метрик
print("\n" + "="*55)
print(f"{'Модальность':12} {'F1':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8}")
print("="*55)
for mod, data in results.items():
    best = max(data["history"], key=lambda h: h["val_f1"])
    print(f"{mod:12} {best['val_f1']:8.4f} {best['val_acc']:8.4f} {best['val_prec']:8.4f} {best['val_rec']:8.4f}  (ep {best['epoch']})")
print("="*55)
